In [1]:
import os
import sys
from pathlib import Path

# 1. Բոտին հասկացնում ենք, թե որտեղ է գտնվում մեր հիմնական freqtrade պանակը
# Քանի որ մենք user_data/notebooks-ում ենք, երկու քայլ հետ ենք գնում
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

# 2. Ներմուծում ենք Freqtrade-ի տվյալների բեռնման գործիքը
from freqtrade.data.history import load_pair_history
from freqtrade.resolvers import StrategyResolver

print("Միջավայրը հաջողությամբ պատրաստ է:")

Միջավայրը հաջողությամբ պատրաստ է:


In [2]:
from freqtrade.enums import CandleType

# 1. Հասցեն ուղղում ենք դեպի բուն binance պանակը
real_user_data_dir = project_root / 'user_data'

config = {
    'strategy': 'MySupertrand_V5',
    'user_data_dir': real_user_data_dir,
    'trading_mode': 'futures'
}

# Կանչում ենք ստրատեգիան
strategy = StrategyResolver.load_strategy(config)

# 2. Բեռնում ենք ՖՅՈՒՉԵՐՍԱՅԻՆ BTC տվյալները feather ֆորմատով
dataframe = load_pair_history(
    datadir=real_user_data_dir / 'data' / 'binance',
    timeframe=strategy.timeframe,
    pair='BTC/USDT:USDT',
    data_format='feather',            # 🔥 Փոխեցինք feather-ի
    candle_type=CandleType.FUTURES
)

print(f"Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա {len(dataframe)} ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:")

Հաջողությա՛մբ բեռնվեց: Աղյուսակում կա 244927 ՖՅՈՒՉԵՐՍԱՅԻՆ մոմ:


In [3]:
# 1. Աշխատեցնում ենք քո ռազմավարության populate_indicators ֆունկցիան
dataframe = strategy.populate_indicators(dataframe, {'pair': 'BTC/USDT:USDT'})

# 2. Էկրանին ենք հանում աղյուսակի վերջին 10 տողերը՝ տեսնելու համար նոր սյունակները
dataframe.tail(10)

,date,open,high,low,close,volume,st_line,st_dir
244917,2026-04-30 09:45:00+00:00,76042.0,76159.6,76035.1,76127.8,329.798,75975.975527,1
244918,2026-04-30 09:50:00+00:00,76127.8,76168.0,76100.0,76111.1,203.983,75975.975527,1
244919,2026-04-30 09:55:00+00:00,76111.2,76111.2,76062.4,76062.7,132.436,75975.975527,1
244920,2026-04-30 10:00:00+00:00,76062.7,76125.5,76054.3,76112.6,182.723,75975.975527,1
244921,2026-04-30 10:05:00+00:00,76112.6,76136.6,76071.4,76110.2,99.186,75975.975527,1
244922,2026-04-30 10:10:00+00:00,76110.2,76114.0,76004.0,76014.4,197.746,75975.975527,1
244923,2026-04-30 10:15:00+00:00,76014.3,76046.1,75984.1,75984.6,262.721,75975.975527,1
244924,2026-04-30 10:20:00+00:00,75984.6,76030.7,75976.3,75976.3,233.632,75975.975527,1
244925,2026-04-30 10:25:00+00:00,75976.3,76000.0,75943.3,75952.8,391.208,76214.188490,-1
244926,2026-04-30 10:30:00+00:00,75952.8,76079.6,75952.8,76022.7,467.381,76214.188490,-1


In [4]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import numpy as np
import plotly.graph_objects as go

# 1. Վերցնում ենք աղյուսակի միայն վերջին 500 մոմերը
df_plot = dataframe.iloc[-500:].copy()

# 🔥 Գաղտնիք. Առանձնացնում ենք Supertrend-ի գծերը ըստ ուղղության (1 և -1)
# Օգտագործում ենք np.where. եթե պայմանը չի կատարվում, դնում ենք np.nan (դատարկ)
df_plot['st_up'] = np.where(df_plot['st_dir'] == 1, df_plot['st_line'], np.nan)
df_plot['st_down'] = np.where(df_plot['st_dir'] == -1, df_plot['st_line'], np.nan)

# 2. Ստեղծում ենք գրաֆիկի պատուհանը
fig = go.Figure()

# 3. Ավելացնում ենք Ճապոնական մոմերը
fig.add_trace(go.Candlestick(
    x=df_plot['date'],
    open=df_plot['open'],
    high=df_plot['high'],
    low=df_plot['low'],
    close=df_plot['close'],
    name='Գին (BTC)'
))

# 4. 🔥 Ավելացնում ենք ԱՃՈՂ Supertrend-ի գիծը (ԿԱՆԱՉ)
fig.add_trace(go.Scatter(
    x=df_plot['date'],
    y=df_plot['st_up'],
    mode='lines',
    name='Supertrend Աճ',
    line=dict(color='#00FF00', width=2.5) # Վառ կանաչ
))

# 5. 🔥 Ավելացնում ենք ԱՆԿՈՒՄԱՅԻՆ Supertrend-ի գիծը (ԿԱՐՄԻՐ)
fig.add_trace(go.Scatter(
    x=df_plot['date'],
    y=df_plot['st_down'],
    mode='lines',
    name='Supertrend Անկում',
    line=dict(color='#FF0000', width=2.5) # Վառ կարմիր
))

# 6. Գրաֆիկի ձևավորումը (Մութ թեմա)
fig.update_layout(
    title='BTC/USDT Ֆյուչերս + 2-Գունանի Supertrend',
    yaxis_title='Գին (USDT)',
    xaxis_title='Ամսաթիվ',
    template='plotly_dark',
    xaxis_rangeslider_visible=False
)

# 7. Ցուցադրում ենք գրաֆիկը էկրանին
# fig.show()

# 8. Ցուցադրում ենք գրաֆիկը ավտոմատ բացվող բրաուզերի թաբում
fig.show(renderer="browser")